In [1]:
import pandas as pd
import os
import state
from datetime import datetime
import evaluate
from sentence_transformers import SentenceTransformer, util
from bert_score import score
import tiktoken
from input_utils import initite_session, prompt_input, save_to_chat_history, list_sessions, clear_session_data
from prompt_utils import red_agent, memory_recall, ai_message_extractor, query_reshaper, n_finder, agent_query_history
from rag_utils import connect_to_vector_store, filter_documens_content, retrieve_documents, get_sources
from model_utils import prompt_designer, initialize_model, query_response
from classification_utils import predict_use_memory
import state

d:\git repos\AI_optimization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
folder_path = "Validation"

file_list = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        rel_path = os.path.relpath(os.path.join(root, file), folder_path)
        file_list.append(rel_path)

In [3]:
enc = tiktoken.encoding_for_model("gpt-4")

def count_tokens(text):
    return len(enc.encode(text))

In [4]:
model  = SentenceTransformer("all-MiniLM-L6-v2")

In [5]:
def val_backend(chat_log, session_name, k = 5):
    user_prompt = chat_log['user_prompt']
    if (session_name in state.session_names):
        n = n_finder(session_name)
        q_token = count_tokens(user_prompt)
        rag_query = query_reshaper(user_prompt, session_name, n = n)
        h_token = count_tokens(agent_query_history(session_name, n = n))
        documents = retrieve_documents(rag_query, k = k)
        context_list = filter_documens_content(documents) 
        sources_list = get_sources(documents)
        prompt = prompt_designer(rag_query, context_list, sources_list)
        p_token = count_tokens(str(prompt))
        ans = query_response(prompt)
        tokens = h_token + q_token + p_token
        return ans,tokens
    else:
        q_token = count_tokens(user_prompt)
        documents = retrieve_documents(user_prompt, k = k)
        context_list = filter_documens_content(documents) 
        sources_list = get_sources(documents)
        prompt = prompt_designer(user_prompt, context_list, sources_list)
        p_token = count_tokens(str(prompt))
        ans = query_response(prompt)
        tokens = q_token + p_token
        return ans, tokens

In [6]:
def val_workflow(session_name):
    session_name = session_name
    data = pd.read_csv(f"Validation/{session_name}.csv")
    metrics = pd.DataFrame(columns = ['Precision', 'Recall', 'F1', 'Sentence-Similarity', 'time', 'tokens'])
    for i in range(10):
        prompt = data['Question'][i]
        chat_log = {
            "timestamp": datetime.now().isoformat(),
            "user_prompt": prompt}
        system_log, tokens = val_backend(chat_log, session_name, k = 5)
        save_to_chat_history(chat_log, system_log, session_name)
        metrics.loc[i, 'tokens'] = tokens
        refs = [data['Answer'][i]]
        cands = [system_log["system_response"]]

        sim = util.cos_sim(
            model.encode(refs, convert_to_tensor=True),
            model.encode(cands, convert_to_tensor=True)
            )
        metrics.loc[i, 'Sentence-Similarity'] = sim.item()

        P, R, F1 = score(
            cands, refs, lang="en", verbose=True)
        metrics.loc[i,'Precision'] = P.item()
        metrics.loc[i, 'Recall'] = R.item()
        metrics.loc[i, 'F1'] = F1.item()

        qt = str(chat_log["timestamp"])
        at = str(system_log["timestamp"])

        t1 = datetime.fromisoformat(qt)
        t2 = datetime.fromisoformat(at)

        metrics.loc[i, 'time'] = (t2-t1)
        print(metrics.loc[i])
    
    return metrics

In [7]:
metrics1 = val_workflow('Conversation1')
metrics2 = val_workflow('Conversation2')
metrics3 = val_workflow('Conversation3')
metrics4 = val_workflow('Conversation4')
metrics5 = val_workflow('Conversation5')
metrics6 = val_workflow('Conversation6')
metrics7 = val_workflow('Conversation7')
metrics8 = val_workflow('Conversation8')
metrics9 = val_workflow('Conversation9')
metrics10 = val_workflow('Conversation10')

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 311.54it/s]


...

calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:05<00:00,  5.54s/it]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 174.97it/s]


done in 5.56 seconds, 0.18 sentences/sec
Precision                    0.772199
Recall                       0.890826
F1                           0.827282
Sentence-Similarity          0.477035
time                   0:00:16.235643
tokens                           3021
Name: 9, dtype: object


In [8]:
metrics = pd.concat(
    [metrics1, metrics2, metrics3, metrics4, metrics5, metrics6, metrics7, metrics8, metrics9, metrics10],
    ignore_index=True
)

In [ ]:
metrics.to_csv('Validation/metrics2.csv')

In [10]:
metrics['time'] = pd.to_timedelta(metrics['time'])
metrics.mean()

Precision                               0.698397
Recall                                  0.793071
F1                                      0.742521
Sentence-Similarity                      0.59076
time                   0 days 00:00:18.131450950
tokens                                   2852.54
dtype: object